In [1]:
import re
import shutil
import statistics
from pathlib import Path
import numpy as np
import tifffile
import mrcfile

#### data/store path

In [26]:
src_root = Path(r"C:\Users\luoyi\Desktop\Data\PdSe2\3DED\20260324")
dst_root = Path(r"C:\Users\luoyi\Desktop\Data\PdSe2\3DED\20260324\20260324_3DED")

#### experiment parameters

In [29]:
WAVELENGTH = 0.025079
CCDPIXELSIZE = 0.007249
ROTATIONAXIS = 117.4
BEAMTILTSTEP = 0
BEAMTILTRANGE = 0.0
STRETCHINGMP = 0.0
STRETCHINGAZIMUTH = 0.0

#### data reorganization

In [32]:
delay_pattern = re.compile(r"([0-9]+(?:\.[0-9]+)?)ps")

In [34]:
tiff_files = list(src_root.rglob("*.tiff"))

delay_map = {}
for f in tiff_files:
    m = delay_pattern.search(f.name)
    if not m:
        continue
    delay = float(m.group(1))
    delay_map.setdefault(delay, []).append(f)

sorted_delays = sorted(delay_map.keys())
delay_to_exp = {d: f"experiment_{i+1}" for i, d in enumerate(sorted_delays)}

##### to .ed3d file

In [37]:
for delay in sorted_delays:
    exp = delay_to_exp[delay]
    exp_dir = dst_root / exp
    tiff_out = exp_dir / "tiff"
    red_out = exp_dir / "RED"

    tiff_out.mkdir(parents=True, exist_ok=True)
    red_out.mkdir(parents=True, exist_ok=True)

    # ---- 收集并按角度排序 ----
    angle_files = []
    for f in delay_map[delay]:
        angle = float(f.parent.name)
        angle_files.append((angle, f))
    angle_files.sort(key=lambda x: x[0])

    # ---- 写 angles.txt + 拷贝 TIFF + 生成 MRC ----
    angles = []
    with (exp_dir / "angles.txt").open("w", encoding="utf-8") as af:
        for idx, (angle, src_tiff) in enumerate(angle_files, start=1):
            name = f"{idx:05d}"
            angles.append(angle)

            # TIFF
            shutil.copy2(src_tiff, tiff_out / f"{name}.tiff")

            # MRC
            img = tifffile.imread(src_tiff)

            if img.dtype == np.uint16:
                img_u16 = img
            else:
                img_u16 = np.clip(img, 0, 65535).astype(np.uint16)
            
            with mrcfile.new(red_out / f"{name}.mrc", overwrite=True) as mrc:
                mrc.set_data(img_u16)

    # ---- 自动推断几何参数 ----
    diffs = [
        angles[i+1] - angles[i]
        for i in range(len(angles) - 1)
        if abs(angles[i+1] - angles[i]) > 1e-6
    ]
    stepsize = statistics.median(diffs) if diffs else 0.0

    # ---- 写 experiment_info.txt ----
    with (exp_dir / "experiment_info.txt").open("w", encoding="utf-8") as f:
        f.write(
f"""Experiment: {exp}
Delay time: {delay:.3f} ps

Auto-derived parameters:
Start angle: {angles[0]:.2f} degrees
End angle: {angles[-1]:.2f} degrees
Rotation range: {angles[-1] - angles[0]:.2f} degrees
Stepsize: {stepsize:.4f} degrees
Number of frames: {len(angles)}

(Other experimental parameters to be filled manually)
"""
        )

    # ---- 直接写 RED/.ed3d（FILELIST = .mrc）----
    with (red_out / f"{exp}.ed3d").open("w", encoding="utf-8") as fout:
        fout.write(f"WAVELENGTH    {WAVELENGTH:.6f}\n")
        fout.write(f"ROTATIONAXIS    {ROTATIONAXIS:.6f}\n")
        fout.write(f"CCDPIXELSIZE    {CCDPIXELSIZE:.6f}\n")
        fout.write(f"GONIOTILTSTEP    {stepsize:.6f}\n")
        fout.write(f"BEAMTILTSTEP    {BEAMTILTSTEP}\n")
        fout.write(f"BEAMTILTRANGE    {BEAMTILTRANGE:.3f}\n")
        fout.write(f"STRETCHINGMP    {STRETCHINGMP:.1f}\n")
        fout.write(f"STRETCHINGAZIMUTH    {STRETCHINGAZIMUTH:.1f}\n\n")

        fout.write("FILELIST\n")
        for idx, angle in enumerate(angles, start=1):
            fout.write(
                f"FILE {idx:05d}.mrc        {angle:8.4f}    0        {angle:8.4f}\n"
            )
        fout.write("ENDFILELIST\n")

# ================= 3. 写 delay_map.txt =================
with (dst_root / "delay_map.txt").open("w", encoding="utf-8") as f:
    for d in sorted_delays:
        f.write(f"{delay_to_exp[d]}\t{d:.3f}\n")

print("Pipeline finished successfully.")

Pipeline finished successfully.


##### to SMV file

In [ ]:
import numpy as np
import tifffile
from pathlib import Path

# ========= 需要你填写 =========
tiff_dir = Path(r"C:\path\to\experiment_1\tiff")
d3_file = Path(r"C:\path\to\experiment_1\RED\experiment_1.3ded")
output_dir = Path(r"C:\path\to\experiment_1\XDS")

DISTANCE = float(input("Enter camera length (mm): "))
# =================================

PIXEL_SIZE_MM = 0.055  # 55 µm
HEADER_BYTES = 1024

output_dir.mkdir(parents=True, exist_ok=True)


# ===== 读取 3ded 角度 =====
def read_3ded(d3_path):
    angles = []
    wavelength = None

    with open(d3_path, "r") as f:
        for line in f:
            if line.startswith("WAVELENGTH"):
                wavelength = float(line.split()[1])
            if line.startswith("FILE"):
                parts = line.split()
                angles.append(float(parts[2]))

    if len(angles) < 2:
        raise ValueError("Not enough frames.")

    osc_start = angles[0]
    osc_range = np.mean(np.diff(angles))

    return osc_start, osc_range, wavelength


OSC_START, OSC_RANGE, WAVELENGTH = read_3ded(d3_file)

print("Detected:")
print("OSC_START =", OSC_START)
print("OSC_RANGE =", OSC_RANGE)
print("WAVELENGTH =", WAVELENGTH)


# ===== TIFF → IMG =====
tiff_files = sorted(tiff_dir.glob("*.tiff"))

for i, tiff_path in enumerate(tiff_files, start=1):

    img_data = tifffile.imread(tiff_path)

    if img_data.dtype != np.uint16:
        img_data = np.clip(img_data, 0, 65535).astype(np.uint16)

    ny, nx = img_data.shape
    osc_angle = OSC_START + (i - 1) * OSC_RANGE

    header = f"""{{ 
HEADER_BYTES= {HEADER_BYTES};
DIM=2;
BYTE_ORDER=little_endian;
TYPE=unsigned_short;
SIZE1={nx};
SIZE2={ny};
PIXEL_SIZE={PIXEL_SIZE_MM};
DISTANCE={DISTANCE};
WAVELENGTH={WAVELENGTH};
OSC_START={osc_angle};
OSC_RANGE={OSC_RANGE};
BEAM_CENTER_X={nx/2};
BEAM_CENTER_Y={ny/2};
}}
"""

    header_bytes = header.encode('ascii')
    padding = HEADER_BYTES - len(header_bytes)
    header_bytes += b' ' * padding

    out_file = output_dir / f"{i:05d}.img"

    with open(out_file, "wb") as f:
        f.write(header_bytes)
        img_data.astype('<u2').tofile(f)


# ===== 自动生成 XDS.INP =====
xds_inp = f"""
JOB= XYCORR INIT COLSPOT IDXREF DEFPIX INTEGRATE CORRECT

NAME_TEMPLATE_OF_DATA_FRAMES= 000??.img
DATA_RANGE= 1 {len(tiff_files)}
SPOT_RANGE= 1 {len(tiff_files)}

DETECTOR= ADSC
NX= {nx}
NY= {ny}
QX= {PIXEL_SIZE_MM}
QY= {PIXEL_SIZE_MM}

OSCILLATION_RANGE= {OSC_RANGE}
X-RAY_WAVELENGTH= {WAVELENGTH}
DETECTOR_DISTANCE= {DISTANCE}

ROTATION_AXIS= 1 0 0
INCIDENT_BEAM_DIRECTION= 0 0 1

FRACTION_OF_POLARIZATION= 0
POLARIZATION_PLANE_NORMAL= 0 1 0

OVERLOAD= 65000
TRUSTED_REGION= 0.0 1.41
"""

with open(output_dir / "XDS.INP", "w") as f:
    f.write(xds_inp)

print("Conversion + XDS.INP generation complete.")